In [ ]:
# 환경 (MiniCarEnv)
# - 상태: [x, y, θ, v]
#   - 행동: 0=좌, 1=직진, 2=우
#   - 보상: 도로 중심선(y=50)과의 거리 벌점
#           x>90 도달 시 보상 +10, 에피소드 종료
#           도로 이탈 시 감점 -10, 종료,  그 외는 보상 +1
#  DQN 모델 정의
#   - Dense(64) × 2(은닉층) → Linear 출력층
#   - MSE loss, Adam optimizer
#  학습 루프
#   epsilon-greedy 정책
#   experience replay (replay buffer: deque)
#   target network 주기적 동기화

# pip install gymnasium

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import animation
from tensorflow import keras
from collections import deque
import random
import gymnasium as gym
from gymnasium import spaces  # 강화학습 환경 정의시 사용할 상태공간과 행동공간 설정용

In [ ]:
# 환경 : 미니카가 도로를 달리며 목적지에 도달하고 도로 중앙선(y=50)에 가까이 있도록 학습하게끔 설계
class MiniCarEnv(gym.Env):
    def __init__(self):
        super(MiniCarEnv, self).__init__()
        # 상태 공간 : 상태는 [x, y, θ, v]의 4차원 벡터
        # x: 수평위치 (0~100), y: 수직위치 (도로중앙은 y=50), θ:방향 각도(-π ~ +π), v:속도(0~5)
        # 이 공간에서 미니카의 현재 상태를 표현
        self.observation_space = spaces.Box(low=np.array([0, 0, -np.pi, 0], dtype=np.float32), \
                    high = np.array([100, 100, np.pi, 5], dtype=np.float32))
        # 행동은 3가지 : 0(좌), 1(직진), 2(우)
        self.action_space = spaces.Discrete(3)
        self.reset()

    # 에피소드 시작 시 초기 상태를 설정
    def reset(self, seed=None, options=None):
        x = 10.0      # 왼쪽에서 시작
        y = 50.0      # 도로 중앙
        theta = 0.0   # θ 오른쪽 정면
        v = 1.0       # 속도
        self.state = np.array([x, y, theta, v])
        return self.state, {}  # {}는 빈 딕셔너리, Gymnasium의 표준반환형식을 따르기 위해 포함

    # 한 개의 타임 스텝 진행 함수
    def step(self, action):
        # self.state는 NumPy 배열로, 미니카의 현재 상태를 나타냄.
        x, y, theta, v = self.state.astype(np.float64)

        # 조향 업데이트 ---
        steer_step = 0.10
        if action == 0:            # 왼쪽 조향(방향을 조금 조절)
            theta -= steer_step    # theta를 -0.1만큼 줄이면 → 왼쪽으로 약간 꺾음
        elif action == 2:          # 오른쪽 조향
            theta += steer_step    # theta를 +0.1만큼 늘리면 → 오른쪽으로 약간 꺾음

        theta *= 0.98         # 조향 감쇠(자연스럽게 직진으로 돌아오게), 과도한 선회 억제
        theta = (theta + np.pi) % (2 * np.pi) - np.pi # 각도 래핑:[-pi, pi]로 유지(무한히 커지지 않게)
        # 각도를 향상 [-pi, pi] (-180 ~ 180) 범위 안에 맞추기

        # 이동 ---
        n = np.random.normal(0, 0.02, size=2)  # 강화학습에 반영할 노이즈(센서오차, 바람, 미끄러짐 등)
        x_prev = x

        # 조향 조정 후 방향을 따라 이동(theta를 반영해 이동)
        # 물체가 속도 V로 방향 θ를 향해 움직일 때의 x축 이동량
        x = x + v * np.cos(theta) + n[0]

        y = y + v * np.sin(theta) + n[1] # 이 두 줄로 θ방향을 기준으로 한 칸 이동하는 것
        # 예: 처음 theta = 0이면 → 오른쪽(x축 방향)으로 직진,
        # theta += 0.1 하면 → 살짝 오른쪽 위 대각선 방향, 계속 theta += 0.1 하면
        # 점점 위쪽으로 커브를 그리며 이동. 에이전트는 불확실한 환경에서도 잘 작동하도록 학습한다,

        self.state = np.array([x, y, theta, v], dtype=np.float32)  # 상태 업데이트

        # 보상 설계 : 중앙선 패널티(완만): 편차 완화. 이런 보상 설계는 중앙산 근처로 주행을 유도하기 위한 것임
        center_penalty = -0.05 * abs(y - 50.00)  # y축 중앙선(50)에서 멀어질수록 감점, 도로 중앙 y=50을 기준으로 변경

        # 진행 보상: 앞으로 간 만큼 보상 (뒤로/옆걸음 억제)
        progress = max(0.0, x - x_prev) * 0.8
        alive = 0.2   # 생존 보상은 작게 (항상 +1.0은 과함)

        reward = alive + center_penalty + progress

        # 종료 조건
        terminated = False
        truncated = False

        if x > 90 and 0 <= y <= 100:    # 목적 달성인 경우 최고 보상 후 종료
            reward += 50.0
            terminated = True
        elif not (0 <= x <= 100 and 0 <= y <= 100):
            reward -= 15.0
            terminated = True

        return self.state, float(reward), terminated, truncated, {}
        # 반환 형식은 Gymnasium 표준: (next_state, reward, done, truncated, info)
        # truncated=False: 시간제한 종료가 아님, info={}: 추가 정보 없음

# 요약 :위 환경은 다음을 유도한다.
# x 방향으로 이동하여 90 이상 도달하면 성공, 도로 중앙선(y=50)을 유지하면서 이동할수록 더 많은 보상.
# 도로를 이탈하거나 목적지 도달 시 에피소드 종료.


In [ ]:
# “성공률” 체크하는 간단 지표 : 성공 여부는 "ep 끝에서 x>90"로 판정
# 이 그래프가 우상향하고, 성공률이 70%+로 올라가면 “안정”으로 봐도 좋다.
success_flags = [traj_end[0] > 90 for _, traj in run_trajectories for traj_end in [traj[-1]]]
success_rate = sum(success_flags) / len(success_flags)
print(f"Success rate: {success_rate*100:.1f}%")

# 이동 평균 보상(예: 20-ep window)
def moving_avg(a, w=20):
    return np.convolve(a, np.ones(w)/w, mode='valid')

w = 20
ma = moving_avg(reward_history, w)
episodes = np.arange(w-1, w-1 + len(ma))  # 에피소드 번호 정렬

plt.figure(figsize=(6,3))
plt.plot(episodes, ma, label=f"Moving Avg (w={w})")
plt.title("Moving Avg Reward (window=20)")
plt.xlabel("Episode")  
plt.ylabel("Reward (moving average)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

"""
이 그래프는 매우 만족스러운 학습 결과를 보여주는 전형적인 DQN 패턴이다.
"""


# 편차(Deviation) 이동평균 그래프 → 중앙선을 얼마나 잘 유지하는지 시각적으로 확인 가능.
plt.plot(moving_avg(deviation_history, 20))
plt.title("Moving Avg Deviation (lower=better)")
plt.xlabel("Episode")
plt.ylabel("Mean deviation (|y-50|)")
plt.show()